# Full CTR model benchmark on Kaggle 2xT4

This notebook synchronizes the evaluation branch, builds feature-engineered parquet files from the processed Kaggle input, trains all five CTR models sequentially, and evaluates validation/test predictions. It defaults to the full dataset (FIT_SAMPLE_SIZE = 0) and writes resumable artifacts, metrics, and plots.

The notebook does not commit or push generated artifacts. Model files are intentionally excluded from the result bundle because they are large.

In [ ]:
import importlib.metadata as metadata
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/CTR-prediction-on-Alibaba-Display-Advertising-Dataset.git"
BRANCH = "feat/evalutation-and-build-experiments"
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
ON_KAGGLE = Path("/kaggle/input").exists()

def run_command(command, cwd=None, redact=()):
    shown = " ".join(str(item) for item in command)
    for secret in redact:
        shown = shown.replace(str(secret), "***")
    print("$", shown)
    return subprocess.run(command, cwd=cwd, check=True)

if ON_KAGGLE:
    repo_url = REPO_URL
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
        if token:
            repo_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        print("No GitHub token found; trying the repository without credentials.")

    ROOT = Path("/kaggle/working") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if (ROOT / ".git").exists():
        run_command(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], redact=(token,) if token else ())
        run_command(["git", "-C", str(ROOT), "switch", BRANCH], redact=(token,) if token else ())
        run_command(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", BRANCH], redact=(token,) if token else ())
    else:
        run_command(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, str(ROOT)], redact=(token,) if token else ())
else:
    ROOT = Path.cwd()
    while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
GIT_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_info = "nvidia-smi unavailable"
gpu_count = len([line for line in gpu_info.splitlines() if line.strip()]) if gpu_info != "nvidia-smi unavailable" else 0
if ON_KAGGLE and gpu_count < 2:
    raise RuntimeError(f"Expected two Kaggle GPUs, detected {gpu_count}: {gpu_info}")
try:
    import psutil
    memory_info = f"{psutil.virtual_memory().total / 1e9:.1f} GB RAM"
except ImportError:
    memory_info = "RAM information unavailable"
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Repository: {ROOT}\nCommit: {GIT_COMMIT}")
print(f"GPU(s): {gpu_info}\n{memory_info}")

In [ ]:
try:
    from packaging.requirements import Requirement
    from packaging.version import Version
except ImportError:
    run_command([sys.executable, "-m", "pip", "install", "-q", "packaging>=23"])
    from packaging.requirements import Requirement
    from packaging.version import Version

REQUIRED = {
    "numpy": "numpy>=1.24", "pandas": "pandas>=2.0",
    "polars": "polars>=0.20", "pyarrow": "pyarrow>=14",
    "scipy": "scipy>=1.10", "sklearn": "scikit-learn>=1.3",
    "joblib": "joblib>=1.3", "psutil": "psutil>=5.9", "yaml": "pyyaml>=6.0",
    "lightgbm": "lightgbm>=4.0", "xgboost": "xgboost>=2.0",
    "catboost": "catboost>=1.2", "matplotlib": "matplotlib>=3.7",
}
missing = []
for module, requirement in REQUIRED.items():
    try:
        parsed = Requirement(requirement)
        installed = Version(metadata.version(parsed.name))
        if parsed.specifier and installed not in parsed.specifier:
            missing.append(requirement)
    except metadata.PackageNotFoundError:
        missing.append(requirement)
if missing:
    print("Installing missing packages:", missing)
    run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import gc
import json
import time
import polars as pl

def find_input_dir():
    roots = [Path("/kaggle/input")] if ON_KAGGLE else [ROOT / "data/processed"]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root] if (root / "train.parquet").exists() else list(root.rglob("train.parquet"))
        for candidate in candidates:
            if all((candidate.parent / name).exists() for name in ("val.parquet", "test.parquet")):
                return candidate.parent
    raise FileNotFoundError("Could not find train.parquet, val.parquet and test.parquet in Kaggle Input.")

INPUT_DIR = find_input_dir()
FE_DIR = Path("/kaggle/working/ctr_processed_fe") if ON_KAGGLE else ROOT / "data/processed"
FE_DIR.mkdir(parents=True, exist_ok=True)
FE_FILES = [FE_DIR / name for name in ("train_fe.parquet", "val_fe.parquet", "test_fe.parquet")]
metadata_path = FE_DIR / "feature_metadata.json"
def engineered_cache_valid():
    if not all(path.exists() for path in FE_FILES) or not metadata_path.exists():
        return False
    try:
        with metadata_path.open(encoding="utf-8") as handle:
            cached = json.load(handle)
        expected_rows = {
            "train": int(pl.scan_parquet(INPUT_DIR / "train.parquet").select(pl.len()).collect().item()),
            "val": int(pl.scan_parquet(INPUT_DIR / "val.parquet").select(pl.len()).collect().item()),
            "test": int(pl.scan_parquet(INPUT_DIR / "test.parquet").select(pl.len()).collect().item()),
        }
        return all(cached.get(f"num_{split}_rows") == rows for split, rows in expected_rows.items()) and bool(cached.get("columns"))
    except Exception as exc:
        print(f"Ignoring invalid feature-engineering cache: {exc}")
        return False

if not engineered_cache_valid():
    print(f"Building engineered partitions from {INPUT_DIR} -> {FE_DIR}")
    run_command([
        sys.executable, "-m", "src.features.run_feature_engineering",
        "--config", str(ROOT / "configs/feature_engineering.yaml"),
        "--input-dir", str(INPUT_DIR), "--output-dir", str(FE_DIR),
    ], cwd=ROOT)
else:
    print(f"Using existing engineered partitions in {FE_DIR}")
for path in FE_FILES:
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"{path.name}: {path.stat().st_size / 1e9:.2f} GB")
with metadata_path.open(encoding="utf-8") as handle:
    FEATURE_METADATA = json.load(handle)
print("Feature metadata columns:", len(FEATURE_METADATA.get("columns", [])))

In [ ]:
import copy
import yaml

FIT_SAMPLE_SIZE = 0
EVAL_SAMPLE_SIZE = 0
RANDOM_SEED = 42
THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.50]
MODELS_TO_RUN = ["logistic_regression", "lightgbm", "xgboost", "catboost", "random_forest"]
GPU_AVAILABLE = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
USE_GPU = ON_KAGGLE and GPU_AVAILABLE

MODELS_DIR = ROOT / "models"
EXPERIMENTS_DIR = ROOT / "experiments"
PLOTS_DIR = ROOT / "outputs" / "model_evaluation"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

def prepare_config(model_key):
    path = ROOT / "configs" / f"{model_key}.yaml"
    with path.open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}
    config["source_path"] = str(path)
    config["paths"]["processed_dir"] = str(FE_DIR)
    config["paths"]["models_dir"] = str(MODELS_DIR)
    config["paths"]["manifest_output"] = str(EXPERIMENTS_DIR / f"{model_key}_run.json")
    config["data"]["use_fe"] = True
    config["data"]["sample_size"] = FIT_SAMPLE_SIZE
    config["data"]["sample_fraction"] = None
    config["data"]["random_seed"] = RANDOM_SEED
    if model_key == "logistic_regression" and FIT_SAMPLE_SIZE == 0:
        config["params"].update({"use_sgd": True, "solver": "sgd"})
    if model_key == "xgboost" and USE_GPU:
        config["params"]["device"] = "cuda:0"
    if model_key == "catboost" and USE_GPU:
        config["params"].update({"task_type": "GPU", "devices": "0:1", "rsm": 1.0})
    return config

CONFIGS = {model_key: prepare_config(model_key) for model_key in MODELS_TO_RUN}
print("Models:", list(CONFIGS))
print("Full-data training:", FIT_SAMPLE_SIZE == 0, "GPU overrides:", USE_GPU)

In [ ]:
from src.evaluate.evaluator import evaluate_predictions, write_evaluation_outputs
from src.models.data_utils import load_ctr_partition
from src.models.train import fit_from_config, get_model_class

TRAIN_ROWS = int(pl.scan_parquet(FE_DIR / "train_fe.parquet").select(pl.len()).collect().item())
results = []
errors = {}

def artifact_path(model_key):
    config = CONFIGS[model_key]
    basename = config["paths"].get("model_basename", model_key)
    return MODELS_DIR / f"{basename}_fe.joblib"

def manifest_path(model_key):
    return EXPERIMENTS_DIR / f"{model_key}_run.json"

def valid_checkpoint(model_key):
    artifact, manifest = artifact_path(model_key), manifest_path(model_key)
    if not artifact.exists() or not manifest.exists():
        return False
    try:
        with manifest.open(encoding="utf-8") as handle:
            info = json.load(handle)
        get_model_class(model_key).load(artifact)
        return int(info.get("train_rows", -1)) == TRAIN_ROWS
    except Exception as exc:
        print(f"Ignoring invalid checkpoint for {model_key}: {exc}")
        return False

for model_key in MODELS_TO_RUN:
    started, run = time.time(), None
    try:
        config, artifact, manifest = CONFIGS[model_key], artifact_path(model_key), manifest_path(model_key)
        if valid_checkpoint(model_key):
            print(f"[{model_key}] valid full-data checkpoint found; skipping fit")
            model = get_model_class(model_key).load(artifact)
            X_val, y_val = load_ctr_partition(str(FE_DIR), "val", model.feature_names, use_fe=True, sample_size=EVAL_SAMPLE_SIZE, random_seed=RANDOM_SEED)
            X_test, y_test = load_ctr_partition(str(FE_DIR), "test", model.feature_names, use_fe=True, sample_size=EVAL_SAMPLE_SIZE, random_seed=RANDOM_SEED)
            p_val, p_test = model.predict_proba(X_val), model.predict_proba(X_test)
        else:
            print(f"[{model_key}] fitting full dataset")
            run = fit_from_config(
                config=config, config_path=config["source_path"], model_key=model_key,
                processed_dir=str(FE_DIR), models_dir=str(MODELS_DIR),
                sample_size=FIT_SAMPLE_SIZE, sample_fraction=None, random_seed=RANDOM_SEED,
                save_artifact=True, write_manifest=True,
            )
            model = run.model
            X_val, y_val = run.dataset.X_val, run.dataset.y_val
            X_test, y_test = run.dataset.X_test, run.dataset.y_test
            p_val, p_test = model.predict_proba(X_val), model.predict_proba(X_test)
        result = evaluate_predictions(
            model_key, y_val, p_val, y_test, p_test, THRESHOLDS,
            artifact_path=str(artifact), manifest_path=str(manifest),
            metadata={"git_commit": GIT_COMMIT, "fit_sample_size": FIT_SAMPLE_SIZE, "eval_sample_size": EVAL_SAMPLE_SIZE, "gpu": USE_GPU, "elapsed_seconds": round(time.time() - started, 2)},
        )
        results = [item for item in results if item.model_key != model_key] + [result]
        write_evaluation_outputs(
            results, EXPERIMENTS_DIR, PLOTS_DIR,
            metadata={"git_commit": GIT_COMMIT, "processed_dir": str(FE_DIR), "input_dir": str(INPUT_DIR), "fit_sample_size": FIT_SAMPLE_SIZE, "eval_sample_size": EVAL_SAMPLE_SIZE, "random_seed": RANDOM_SEED, "gpu": USE_GPU, "models": MODELS_TO_RUN},
        )
        print(f"[{model_key}] complete in {time.time() - started:.1f}s; val ROC-AUC={result.validation.probability_metrics['roc_auc']:.5f}")
    except Exception as exc:
        errors[model_key] = repr(exc)
        print(f"[{model_key}] FAILED: {exc}")
    finally:
        gc.collect()

if errors:
    raise RuntimeError(f"Model runs failed: {errors}")
print(f"Completed {len(results)} model evaluations.")

In [ ]:
import zipfile

ranking = sorted(results, key=lambda item: (-item.validation.probability_metrics["roc_auc"], item.validation.probability_metrics["log_loss"]))
print("Validation ranking:")
for position, result in enumerate(ranking, start=1):
    metrics = result.validation.probability_metrics
    print(f"{position}. {result.model_key:20s} ROC-AUC={metrics['roc_auc']:.5f} LogLoss={metrics['log_loss']:.5f} PR-AUC={metrics['pr_auc']:.5f}")

bundle = ROOT / "evaluation_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [EXPERIMENTS_DIR / "model_evaluation_results.json", EXPERIMENTS_DIR / "model_metrics.csv", EXPERIMENTS_DIR / "threshold_metrics.csv"]:
        if path.exists():
            archive.write(path, path.relative_to(ROOT))
    for path in PLOTS_DIR.glob("*.png"):
        archive.write(path, path.relative_to(ROOT))
    for model_key in MODELS_TO_RUN:
        path = manifest_path(model_key)
        if path.exists():
            archive.write(path, path.relative_to(ROOT))
print(f"Result bundle: {bundle} ({bundle.stat().st_size / 1e6:.2f} MB)")